# OVRO-LWA source metacatalog

Identify sources in OVRO-LWA wide-field FITS images with **PyBDSF**, then fuse per-image catalogs into a **metacatalog** with one entry per unique sky position.

Each FITS file is associated with an **LST hour bin** (e.g. `01h`) and a **color band** (`Full`, `Red`, `Green`, or `Blue`). PyBDSF is run independently on every image.

**Workflow**
1. Discover FITS files and parse LST hour + band from filenames.
2. Run `bdsf.process_image` on each `(lst_hour, band)` image → per-image source catalogs.
3. **LST merge** (per band): cross-match detections across LST hours within each band. One row per source, covering the union of sky seen in any LST hour. Pick the detection whose peak flux is nearest the **median** flux over matching LST images; copy **all** properties from that row.
4. **Band merge**: associate Blue/Green/Red onto Full-band LST-merged sources (beam-sized radius; median-flux pick when several band sources match). Unmatched band-only sources are kept with `origin_band` set.

Reference conventions: `ovro-lwa-portal` (`fits_to_zarr_xradio.py`) and `image-plane-correction` (`source_detection.py`).

In [1]:
from __future__ import annotations

import os
import re
import tempfile
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable, Iterator

import astropy.units as u
import bdsf
import numpy as np
import pandas as pd
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.table import Table

# --- user configuration ---------------------------------------------------
FITS_ROOT = Path("/fast/claw")  # directory containing FITS images (searched recursively)
OUTPUT_DIR = Path("/fast/claw/metacatalog")  # where catalogs are written

# PyBDSF detection parameters (see image-plane-correction/source_detection.py)
BDSF_KW = dict(
    thresh="hard",
    thresh_isl=7.0,
    thresh_pix=4.0,
    atrous_do=False,
    psf_vary_do=False,
    quiet=True,
    ncores=8,
)

# LST hour bins to process (must have Full + color-band FITS for each)
LST_HOURS = ["01h", "02h", "03h"]
ASSOC_BANDS = ("Blue", "Green", "Red")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Filename parsing

Generalized parsing supports several OVRO-LWA naming conventions:

| Convention | Example | LST hour | Band |
|------------|---------|----------|------|
| Deep color products | `I_01h_deep_Taper_R0_Full.fits` | `01h` | `Full` |
| LST color-band (portal ingest) | `Blue_I_10min_..._20250508_LST22h_t0001.fits` | `22h` | `Blue` |
| Parent directory | `.../01h/.../image.fits` | `01h` | from filename prefix |

Bands are one of `Full`, `Blue`, `Green`, `Red`. Additional LST hour bins and bands can be added without code changes as long as filenames follow these patterns.

In [2]:
COLOR_BANDS = ("Full", "Blue", "Green", "Red")
HOUR_DIR_RE = re.compile(r"^(\d{2})h$", re.IGNORECASE)

# Deep wideband color products: I_01h_deep_Taper_R0_Full.fits
DEEP_COLOR_RE = re.compile(
    r"^I_(\d{2})h_.*_(Full|Blue|Green|Red)\.fits$",
    re.IGNORECASE,
)

# Portal lst-color products: Blue_I_..._20250508_LST22h_t0001.fits
LST_COLOR_RE = re.compile(
    r"^(Full|Blue|Green|Red)_I_.*_(\d{8})_LST(\d{1,2})h_(t\d+)\.fits$",
    re.IGNORECASE,
)

# Band prefix fallback: Blue_I_....fits (LST from directory or LSTnnh elsewhere in name)
BAND_PREFIX_RE = re.compile(
    r"^(Full|Blue|Green|Red)_I_.*\.fits$",
    re.IGNORECASE,
)
LST_IN_NAME_RE = re.compile(r"LST(\d{1,2})h", re.IGNORECASE)


@dataclass(frozen=True, slots=True)
class FitsMetadata:
    path: Path
    lst_hour: str  # e.g. "01h"
    band: str  # Full | Blue | Green | Red
    time_key: str | None = None  # optional lst-color time bin key


def _format_lst_hour(hour: int | str) -> str:
    return f"{int(hour):02d}h"


def _lst_from_parents(path: Path) -> str | None:
    for parent in path.parents:
        m = HOUR_DIR_RE.match(parent.name)
        if m:
            return _format_lst_hour(m.group(1))
    return None


def parse_fits_metadata(path: Path) -> FitsMetadata | None:
    """Return LST hour and color band for a FITS path, or None if unrecognized."""
    name = path.name

    m = DEEP_COLOR_RE.match(name)
    if m:
        return FitsMetadata(path=path, lst_hour=_format_lst_hour(m.group(1)), band=m.group(2).title())

    m = LST_COLOR_RE.match(name)
    if m:
        band, ymd, lst_h, t_bin = m.group(1), m.group(2), m.group(3), m.group(4)
        return FitsMetadata(
            path=path,
            lst_hour=_format_lst_hour(lst_h),
            band=band.title(),
            time_key=f"{ymd}_LST{_format_lst_hour(lst_h)}_{t_bin}",
        )

    m = BAND_PREFIX_RE.match(name)
    if m:
        band = m.group(1).title()
        lst_h = None
        m_lst = LST_IN_NAME_RE.search(name)
        if m_lst:
            lst_h = _format_lst_hour(m_lst.group(1))
        else:
            lst_h = _lst_from_parents(path)
        if lst_h is None:
            return None
        return FitsMetadata(path=path, lst_hour=lst_h, band=band)

    return None


def discover_fits_files(root: Path, *, patterns: Iterable[str] = ("*.fits",)) -> list[FitsMetadata]:
    """Recursively find FITS files and parse metadata."""
    found: list[FitsMetadata] = []
    for pattern in patterns:
        for path in sorted(root.rglob(pattern)):
            meta = parse_fits_metadata(path)
            if meta is not None:
                found.append(meta)
    return found

def slot_glob_patterns(lst_hour: str, band: str) -> list[str]:
    patterns = [
        f"**/*_{lst_hour}_*_{band}.fits",
        f"**/I_{lst_hour}_*_{band}.fits",
        f"I_{lst_hour}_*_{band}.fits",
    ]
    if band != "Full":
        patterns.insert(0, f"**/{band}_I_*_LST{lst_hour[0:2]}h_*.fits")
        patterns.insert(1, f"**/{band}_I_*_LST{int(lst_hour[:-1])}h_*.fits")
    else:
        patterns.insert(0, f"**/Full_I_*_LST{lst_hour[0:2]}h_*.fits")
    return patterns


def resolve_fits_slot(root: Path, lst_hour: str, band: str) -> Path:
    """Resolve exactly one FITS path for an (lst_hour, band) slot."""
    for pattern in slot_glob_patterns(lst_hour, band):
        matches = sorted({p.resolve() for p in root.glob(pattern)})
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            raise FileNotFoundError(
                f"Ambiguous glob for ({lst_hour}, {band}): pattern {pattern!r} matched "
                f"{len(matches)} files: {[m.name for m in matches]}"
            )
    raise FileNotFoundError(f"No FITS found for ({lst_hour}, {band}) under {root}")


In [ ]:
fits_files = discover_fits_files(FITS_ROOT)
summary = pd.DataFrame(
    {
        "path": [m.path.name for m in fits_files],
        "lst_hour": [m.lst_hour for m in fits_files],
        "band": [m.band for m in fits_files],
        "time_key": [m.time_key for m in fits_files],
    }
)
print(f"Found {len(fits_files)} FITS files under {FITS_ROOT}")
summary.sort_values(["lst_hour", "band"]).reset_index(drop=True)

## PyBDSF source detection

For each `(lst_hour, band)` image we:
1. Read the FITS primary HDU.
2. Sanitize non-finite pixels and ensure frequency keywords PyBDSF expects (`RESTFREQ`).
3. Pass the HDU directly to `bdsf.process_image`.
4. Export the Gaussian list (`catalog_type='gaul'`) as an Astropy `Table`, then convert to a pandas `DataFrame` with provenance columns.

In [4]:
GAUL_COLUMNS = [
    "RA",
    "DEC",
    "E_RA",
    "E_DEC",
    "Total_flux",
    "E_Total_flux",
    "Peak_flux",
    "E_Peak_flux",
    "Maj",
    "E_Maj",
    "Min",
    "E_Min",
    "PA",
    "E_PA",
    "DC_Maj",
    "DC_Min",
    "DC_PA",
    "S_Code",
    "Gaus_id",
    "Isl_id",
    "Source_id",
]


def _restfreq_hz(header: fits.Header) -> float | None:
    for key in ("RESTFREQ", "RESTFRQ", "CRVAL3", "FREQ"):
        if key in header:
            try:
                return float(header[key])
            except (TypeError, ValueError):
                continue
    return None


def _prepare_hdu(path: Path) -> fits.PrimaryHDU:
    """Read FITS, squeeze to 2D, sanitize NaNs, and fix header for PyBDSF."""
    with fits.open(path, memmap=True) as hdul:
        hdu = hdul[0]
        data = np.squeeze(np.asarray(hdu.data, dtype=np.float32))
        if data.ndim != 2:
            raise ValueError(f"Expected 2D image in {path.name}, got shape {data.shape}")
        data = np.where(np.isfinite(data), data, 0.0)
        header = hdu.header.copy()
    rf = _restfreq_hz(header)
    if rf is not None:
        header["RESTFREQ"] = rf
        header["RESTFRQ"] = rf
    return fits.PrimaryHDU(data=data, header=header)


def _beam_from_header(header: fits.Header) -> tuple[float, float, float]:
    if "BMAJ" not in header or "BMIN" not in header:
        raise ValueError("FITS header missing BMAJ/BMIN beam keywords")
    return (
        float(header["BMAJ"]),
        float(header["BMIN"]),
        float(header.get("BPA", 0.0)),
    )


def run_pybdsf_on_hdu(hdu: fits.PrimaryHDU, **process_kw) -> Table:
    """Run PyBDSF on an in-memory HDU and return the Gaussian catalog table."""
    beam = _beam_from_header(hdu.header)
    kw = dict(BDSF_KW)
    kw.update(process_kw)
    kw["beam"] = beam

    img = bdsf.process_image(hdu, **kw)

    with tempfile.NamedTemporaryFile(suffix=".gaul.fits", delete=False) as tmp:
        cat_path = tmp.name
    try:
        img.write_catalog(outfile=cat_path, format="fits", catalog_type="gaul", clobber=True)
        return Table.read(cat_path)
    finally:
        try:
            os.unlink(cat_path)
        except OSError:
            pass


def detect_sources(meta: FitsMetadata) -> pd.DataFrame:
    """Detect sources in one FITS image; return a catalog DataFrame."""
    hdu = _prepare_hdu(meta.path)
    bmaj, bmin, bpa = _beam_from_header(hdu.header)
    table = run_pybdsf_on_hdu(hdu)
    df = table.to_pandas()

    keep = [c for c in GAUL_COLUMNS if c in df.columns]
    df = df[keep].copy()
    df["lst_hour"] = meta.lst_hour
    df["band"] = meta.band
    df["source_file"] = meta.path.name
    if meta.time_key is not None:
        df["time_key"] = meta.time_key
    df["BMAJ"] = bmaj
    df["BMIN"] = bmin
    df["BPA"] = bpa
    return df

In [ ]:
per_image_catalogs: dict[tuple[str, str], pd.DataFrame] = {}

for lst_hour in LST_HOURS:
    for band in COLOR_BANDS:
        path = resolve_fits_slot(FITS_ROOT, lst_hour, band)
        meta = parse_fits_metadata(path)
        if meta is None:
            meta = FitsMetadata(path=path, lst_hour=lst_hour, band=band)
        key = (lst_hour, band)
        print(f"PyBDSF: {path.name}  (LST={lst_hour}, band={band})")
        catalog = detect_sources(meta)
        per_image_catalogs[key] = catalog

        out_csv = OUTPUT_DIR / f"sources_{lst_hour}_{band}.csv"
        catalog.to_csv(out_csv, index=False)
        print(f"  -> {len(catalog)} sources written to {out_csv}")

len(per_image_catalogs)

## Metacatalog fusion

Two stages:

1. **`merge_lst_metacatalog`** — within each band, fuse detections from all LST hours. Matching the same source across LST bins is expected. The representative row is the detection whose `Peak_flux` is closest to the median over the cluster; all scalar properties come from that single detection.

2. **`build_global_metacatalog`** — fuse the per-band LST-merged catalogs. Full-band rows are masters; Blue/Green/Red sources within a beam-sized radius are associated (median-flux pick when several match). Band-only sources without a Full match are appended with `origin_band` set.

In [ ]:
from astropy.coordinates import angular_separation

BAND_FIELDS = (
    "Peak_flux",
    "Total_flux",
    "RA",
    "DEC",
    "Maj",
    "Min",
    "PA",
    "DC_Maj",
    "DC_Min",
    "DC_PA",
)


def _pick_median_flux_row(df: pd.DataFrame, flux_col: str = "Peak_flux") -> pd.Series:
    """Return the row whose flux is closest to the median over *df*."""
    flux = df[flux_col].to_numpy(dtype=float)
    finite = np.isfinite(flux)
    if not finite.any():
        return df.iloc[0]
    med = float(np.nanmedian(flux[finite]))
    idx = int(np.nanargmin(np.abs(flux - med)))
    return df.iloc[idx]


def _match_radius_deg(bmaj_a: float, bmaj_b: float) -> float:
    return float(max(bmaj_a, bmaj_b))


def _skycoord(df: pd.DataFrame) -> SkyCoord:
    return SkyCoord(ra=df["RA"].to_numpy() * u.deg, dec=df["DEC"].to_numpy() * u.deg)


def _cluster_by_sky_position(
    df: pd.DataFrame,
    *,
    sort_col: str = "Peak_flux",
    bmaj_col: str = "BMAJ",
) -> list[pd.DataFrame]:
    """Greedy beam-sized clustering; return one member DataFrame per cluster."""
    if df.empty:
        return []

    work = df[np.isfinite(df["RA"]) & np.isfinite(df["DEC"])].copy()
    work = work.sort_values(sort_col, ascending=False, na_position="last")

    cluster_ra: list[float] = []
    cluster_dec: list[float] = []
    cluster_bmaj: list[float] = []
    cluster_members: list[list[dict]] = []

    for row in work.itertuples(index=False):
        rd = row._asdict()
        ra_row = float(rd["RA"])
        dec_row = float(rd["DEC"])
        bmaj_row = float(rd.get(bmaj_col, np.nan))
        if not np.isfinite(bmaj_row):
            bmaj_row = 0.0

        best_idx: int | None = None
        if cluster_ra:
            ra2 = np.asarray(cluster_ra, dtype=float)
            dec2 = np.asarray(cluster_dec, dtype=float)
            seps = np.rad2deg(
                angular_separation(
                    np.deg2rad(ra_row),
                    np.deg2rad(dec_row),
                    np.deg2rad(ra2),
                    np.deg2rad(dec2),
                )
            )
            radii = np.maximum(bmaj_row, np.asarray(cluster_bmaj, dtype=float))
            within = seps <= radii
            if within.any():
                candidates = np.where(within)[0]
                best_idx = int(candidates[np.argmin(seps[candidates])])

        if best_idx is None:
            cluster_ra.append(ra_row)
            cluster_dec.append(dec_row)
            cluster_bmaj.append(bmaj_row)
            cluster_members.append([rd])
        else:
            cluster_members[best_idx].append(rd)
            rep = _pick_median_flux_row(pd.DataFrame(cluster_members[best_idx]))
            cluster_ra[best_idx] = float(rep["RA"])
            cluster_dec[best_idx] = float(rep["DEC"])
            cluster_bmaj[best_idx] = max(cluster_bmaj[best_idx], bmaj_row)

    return [pd.DataFrame(members) for members in cluster_members]


def merge_lst_metacatalog(catalogs: Iterable[pd.DataFrame], *, band: str) -> pd.DataFrame:
    """Fuse per-LST detections within one band → one row per source."""
    combined = pd.concat(list(catalogs), ignore_index=True)
    if combined.empty:
        return pd.DataFrame()

    rows: list[dict] = []
    for members in _cluster_by_sky_position(combined):
        rep = _pick_median_flux_row(members)
        entry = rep.to_dict()
        entry["band"] = band
        entry["n_lst_contributions"] = len(members)
        entry["lst_hours"] = ",".join(sorted(members["lst_hour"].unique()))
        entry["representative_lst"] = rep["lst_hour"]
        rows.append(entry)

    meta = pd.DataFrame(rows)
    return meta.sort_values("Peak_flux", ascending=False, na_position="last").reset_index(drop=True)


def _empty_band_cols() -> dict:
    out: dict = {}
    for band in ASSOC_BANDS:
        for field in BAND_FIELDS:
            out[f"{field}_{band}"] = np.nan
        out[f"n_assoc_{band}"] = 0
    return out


def build_global_metacatalog(lst_merged: dict[str, pd.DataFrame]) -> pd.DataFrame:
    """Fuse LST-merged per-band catalogs: Full masters + band associations."""
    full = lst_merged["Full"].copy()
    rows: list[dict] = []
    matched_band_idx: dict[str, set[int]] = {b: set() for b in ASSOC_BANDS}

    for _, frow in full.iterrows():
        bmaj_full = float(frow["BMAJ"])
        entry = {
            "is_full_master": True,
            "origin_band": "Full",
            "RA": frow["RA"],
            "DEC": frow["DEC"],
            "Peak_flux": frow["Peak_flux"],
            "Total_flux": frow["Total_flux"],
            "Maj": frow["Maj"],
            "Min": frow["Min"],
            "PA": frow["PA"],
            "DC_Maj": frow.get("DC_Maj", np.nan),
            "DC_Min": frow.get("DC_Min", np.nan),
            "DC_PA": frow.get("DC_PA", np.nan),
            "BMAJ_full": bmaj_full,
            "n_lst_contributions": int(frow.get("n_lst_contributions", 1)),
            "lst_hours": frow.get("lst_hours", frow.get("lst_hour", "")),
            "representative_lst": frow.get("representative_lst", frow.get("lst_hour", "")),
            "source_file_Full": frow.get("source_file", ""),
        }
        entry.update(_empty_band_cols())
        fsc = SkyCoord(ra=frow["RA"] * u.deg, dec=frow["DEC"] * u.deg)

        for band in ASSOC_BANDS:
            bdf = lst_merged[band]
            if bdf.empty:
                continue
            hits: list[int] = []
            for idx, brow in bdf.iterrows():
                bsc_row = SkyCoord(ra=brow["RA"] * u.deg, dec=brow["DEC"] * u.deg)
                radius = _match_radius_deg(bmaj_full, float(brow["BMAJ"]))
                if fsc.separation(bsc_row).deg > radius:
                    continue
                hits.append(idx)
                matched_band_idx[band].add(idx)
            if not hits:
                continue
            sub = bdf.loc[hits]
            best = _pick_median_flux_row(sub)
            entry[f"n_assoc_{band}"] = len(hits)
            for field in BAND_FIELDS:
                entry[f"{field}_{band}"] = best[field]
        rows.append(entry)

    for band in ASSOC_BANDS:
        bdf = lst_merged[band]
        for idx, brow in bdf.iterrows():
            if idx in matched_band_idx[band]:
                continue
            entry = {
                "is_full_master": False,
                "origin_band": band,
                "RA": brow["RA"],
                "DEC": brow["DEC"],
                "Peak_flux": brow["Peak_flux"],
                "Total_flux": brow["Total_flux"],
                "Maj": brow["Maj"],
                "Min": brow["Min"],
                "PA": brow["PA"],
                "DC_Maj": brow.get("DC_Maj", np.nan),
                "DC_Min": brow.get("DC_Min", np.nan),
                "DC_PA": brow.get("DC_PA", np.nan),
                "BMAJ_full": np.nan,
                "BMAJ_band": float(brow["BMAJ"]),
                "n_lst_contributions": int(brow.get("n_lst_contributions", 1)),
                "lst_hours": brow.get("lst_hours", brow.get("lst_hour", "")),
                "representative_lst": brow.get("representative_lst", brow.get("lst_hour", "")),
                f"source_file_{band}": brow.get("source_file", ""),
            }
            entry.update(_empty_band_cols())
            for field in BAND_FIELDS:
                entry[f"{field}_{band}"] = brow[field]
            entry[f"n_assoc_{band}"] = 1
            rows.append(entry)

    meta = pd.DataFrame(rows)
    meta.insert(0, "meta_id", range(len(meta)))
    return meta.sort_values("Peak_flux", ascending=False, na_position="last").reset_index(drop=True)

In [ ]:
lst_merged: dict[str, pd.DataFrame] = {}

for band in COLOR_BANDS:
    band_catalogs = [per_image_catalogs[(lst, band)] for lst in LST_HOURS if (lst, band) in per_image_catalogs]
    merged = merge_lst_metacatalog(band_catalogs, band=band)
    lst_merged[band] = merged
    out_csv = OUTPUT_DIR / f"metacatalog_lst_{band}.csv"
    merged.to_csv(out_csv, index=False)
    print(f"LST merge ({band}): {len(merged)} sources -> {out_csv}")

metacatalog = build_global_metacatalog(lst_merged)

meta_csv = OUTPUT_DIR / "metacatalog.csv"
meta_fits = OUTPUT_DIR / "metacatalog.fits"
metacatalog.to_csv(meta_csv, index=False)
Table.from_pandas(metacatalog).write(meta_fits, overwrite=True)

n_detections = sum(len(df) for df in per_image_catalogs.values())
print(f"\nGlobal metacatalog: {len(metacatalog)} sources from {n_detections} per-image detections")
print(f"Wrote {meta_csv}")
print(f"Wrote {meta_fits}")
metacatalog.head(10)

In [ ]:
# LST merge yield (Full band)
full_lst = lst_merged["Full"]
multi_lst = full_lst[full_lst["n_lst_contributions"] > 1].sort_values("n_lst_contributions", ascending=False)
print(f"Full-band sources after LST merge: {len(full_lst)}")
print(f"  seen in multiple LST hours: {len(multi_lst)}")
if len(multi_lst):
    display(multi_lst.head(10)[["RA", "DEC", "Peak_flux", "n_lst_contributions", "lst_hours", "representative_lst"]])

# Global band merge
print(f"\nGlobal rows by origin_band:")
print(metacatalog["origin_band"].value_counts())

band_assoc = metacatalog[metacatalog["is_full_master"] & (metacatalog[[f"n_assoc_{b}" for b in ASSOC_BANDS]].max(axis=1) > 0)]
print(f"Full masters with at least one color-band association: {len(band_assoc)}")
metacatalog.head(10)[["meta_id", "RA", "DEC", "origin_band", "Peak_flux", "lst_hours", "n_assoc_Blue", "n_assoc_Green", "n_assoc_Red"]]